In [ ]:
import sys
import os

# Get the absolute path of the current directory
current_dir = os.getcwd()

# Get the parent directory (project root)
# If your notebook is deeper (e.g., notebooks/exploratory/), you might need os.path.dirname() twice
project_root = os.path.dirname(os.path.dirname(current_dir))

# Add project root to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)

# Verify it points to the folder containing 'src'
print(f"Project Root added: {project_root}")

In [ ]:
import torch
import random
import numpy as np
import os
import logging
import matplotlib.pyplot as plt

import src.utils.utils as utils
import src.data.dataloader as dataloader
import src.tasks.taskloader as taskloader
import src.train.training as training

from src.tasks.base_task import BaseTask
from src.neuralizer.lightning_model import LightningModel
from src.dataset.general_dataset import NeuralizerGeneralDataset
from src.dataset.sampler import WeightedTaskSampler

from torch.utils.data import DataLoader
from typing import List
from tqdm import tqdm


In [ ]:
def seed_everything(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

seed_everything(42)


In [ ]:
# <-- UPDATE THESE PATHS -->
CONFIG_PATH = "/configs/config.yaml"
CHECKPOINT_PATH = "/path/to/your/checkpoint.ckpt"     
OUTPUT_PATH = "/assets/figures"


In [ ]:
import re
import os

def extract_version_from_string(path_to_ckpt: str) -> str:
    match = re.search(r"version_\d+", path_to_ckpt)
    return match.group(0) if match else "Unknown_Version"


In [ ]:
CFG = utils.load_config(CONFIG_PATH)

# Device
_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load metadata
dataloader.load_data(CFG)
metadata_df = dataloader.get_metadata_df(
    CFG["dataloader"]["metadata_location"]
)

# Tasks
seen_tasks: List[BaseTask] = taskloader.get_tasks(
    df=metadata_df,
    CFG=CFG["tasks"],
)

unseen_tasks: List[BaseTask] = taskloader.get_test_tasks(
    df=metadata_df,
    CFG=CFG["tasks"],
)

# Load model
model: LightningModel = training.load_model(
    CFG=CFG,
    path_to_checkpoint=CHECKPOINT_PATH,
    cuda_device="cuda",
    use_cpu=not torch.cuda.is_available(),
)

model = model.to(_DEVICE)
model.eval()


In [ ]:
def get_deterministic_prediction(
    task: BaseTask,
    tag: str,
    draw_idx: int,
    base_seed: int = 42
):
    if tag == "seen":
        offset = 0
    elif tag == "unseen":
        offset = 1000
    else:
        offset = 2000

    run_seed = base_seed + offset + draw_idx

    seed_everything(run_seed)

    dataset = NeuralizerGeneralDataset(
        tasks=[task],
        **CFG["training"]["dataset"]
    )

    sampler = WeightedTaskSampler(dataset)

    loader = DataLoader(
        dataset=dataset,
        sampler=sampler,
        batch_size=CFG["training"]["dataloader"]["batch_size"],
        num_workers=0
    )

    seed_everything(run_seed)
    x, y, ctx_in, ctx_out, lossfun, class_names = next(iter(loader))

    x = x.to(_DEVICE)
    ctx_in = ctx_in.to(_DEVICE)
    ctx_out = ctx_out.to(_DEVICE)

    with torch.inference_mode():
        y_pred = model(x, ctx_in, ctx_out)

    return x.cpu(), y.cpu(), y_pred.cpu()


In [ ]:
def to_numpy_img(tensor):
    """
    Convert [C,H,W] torch tensor to [H,W,C] numpy array.
    Assumes 3-channel RGB.
    """
    img = tensor.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)  # ensure valid range
    return img


def plot_and_save_3x9(tasks: List[BaseTask], tag: str):

    version = extract_version_from_string(CHECKPOINT_PATH)

    save_dir = os.path.join(OUTPUT_PATH, version)
    os.makedirs(save_dir, exist_ok=True)

    for task_index, task in enumerate(tasks):

        fig, axes = plt.subplots(3, 9, figsize=(24, 8))

        for draw_idx in range(3):

            x, y, y_pred = get_deterministic_prediction(
                task=task,
                tag=tag,
                draw_idx=draw_idx
            )

            col_offset = draw_idx * 3

            for i in range(3):

                axes[0, col_offset + i].imshow(
                    x[i].permute(1, 2, 0).numpy().clip(0, 1)
                )
                axes[0, col_offset + i].axis("off")

                axes[1, col_offset + i].imshow(
                    y_pred[i].permute(1, 2, 0).numpy().clip(0, 1)
                )
                axes[1, col_offset + i].axis("off")

                axes[2, col_offset + i].imshow(
                    y[i].permute(1, 2, 0).numpy().clip(0, 1)
                )
                axes[2, col_offset + i].axis("off")

        plt.tight_layout()

        # Clean task name (avoid spaces or illegal chars)
        task_name = task.get_task_name().replace(" ", "_")

        filename = f"prediction_{task_name}_{tag}.png"
        save_path = os.path.join(save_dir, filename)

        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"Saved: {save_path}")



In [ ]:
plot_and_save_3x9(seen_tasks, tag="seen")


In [ ]:
plot_and_save_3x9(unseen_tasks, tag="unseen")
